# SQL Analysis
Analyzing SpaceX data using SQL queries

In [1]:
import pandas as pd
import sqlite3

# Create in-memory database
conn = sqlite3.connect(':memory:')
df = pd.read_csv('spacex_launches_raw.csv')
df.to_sql('launches', conn, index=False, if_exists='replace')

### Query 1: Unique Launch Sites

In [2]:
query1 = "SELECT DISTINCT Launch_Site FROM launches ORDER BY Launch_Site"
sites = pd.read_sql_query(query1, conn)

print("Unique Launch Sites:")
for idx, row in sites.iterrows():
    print(f"{idx+1}. {row['Launch_Site']}")

Unique Launch Sites:
1. CCAFS LC-40
2. VAFB SLC-4E
3. KSC LC-39A
4. Boca Chica

### Query 2: Total Payload Mass

In [3]:
query2 = "SELECT SUM(Payload_Mass_kg) as total_mass FROM launches"
total = pd.read_sql_query(query2, conn)
print(f"Total Payload Mass Delivered: {total['total_mass'][0]:,.0f} kg")

Total Payload Mass Delivered: 240,294 kg

### Query 3: Average Payload by Site

In [4]:
query3 = "SELECT Launch_Site, ROUND(AVG(Payload_Mass_kg), 0) as avg_mass FROM launches GROUP BY Launch_Site ORDER BY Launch_Site"
avg_by_site = pd.read_sql_query(query3, conn)

print("Average Payload by Site:")
for _, row in avg_by_site.iterrows():
    print(f"{row['Launch_Site']}: {row['avg_mass']:,.0f} kg")

### Query 4: Success Rate by Site

In [5]:
query4 = """SELECT Launch_Site, COUNT(*) as total, SUM(Class) as successes, 
            ROUND(SUM(Class)*100.0/COUNT(*), 0) as success_rate 
            FROM launches GROUP BY Launch_Site ORDER BY success_rate DESC"""
success_by_site = pd.read_sql_query(query4, conn)

print("Success Rate by Launch Site:")
for _, row in success_by_site.iterrows():
    print(f"{row['Launch_Site']}: {int(row['success_rate'])}%")

### Query 5: Booster Reuse Statistics

In [6]:
query5 = "SELECT AVG(Booster_Landings) as avg_landings, MAX(Booster_Landings) as max_landings FROM launches"
booster_stats = pd.read_sql_query(query5, conn)

print(f"Booster Experience Statistics:")
print(f"Average landings per booster: {booster_stats['avg_landings'][0]:.1f}")
print(f"Max landings by single booster: {int(booster_stats['max_landings'][0])}")

### Query 6: Failure Analysis by Orbit

In [7]:
query6 = """SELECT Orbit_Type, COUNT(*) as total, SUM(CASE WHEN Class=0 THEN 1 ELSE 0 END) as failures,
            ROUND(SUM(CASE WHEN Class=0 THEN 1 ELSE 0 END)*100.0/COUNT(*), 0) as failure_rate
            FROM launches GROUP BY Orbit_Type ORDER BY failure_rate DESC"""
failures_by_orbit = pd.read_sql_query(query6, conn)

print("Failure Rate by Orbit Type:")
for _, row in failures_by_orbit.iterrows():
    print(f"{row['Orbit_Type']}: {int(row['failure_rate'])}%")